In [2]:
# =========================================================
# RAG Bot V3.0 - Streamlit App
# Hybrid Retrieval (BM25 + ChromaDB) + Conversational Memory
# =========================================================

# -------------------------------
# SETUP (run once before deploying)
# -------------------------------
# pip install streamlit langchain-community langchain-text-splitters chromadb sentence-transformers pypdf rank-bm25 langchain-google-genai langchain-core

import streamlit as st
import os

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage


# -------------------------------
# PAGE CONFIG
# -------------------------------
st.set_page_config(page_title="RAG Bot V3.0", page_icon="🤖", layout="centered")
st.title("🤖 RAG Bot V3.0")
st.caption("Hybrid Retrieval (BM25 + ChromaDB) with Conversational Memory")


# -------------------------------
# API KEY HANDLING
# -------------------------------
# For Hugging Face Spaces / Streamlit Cloud: set this as a "Secret" in the
# app settings, named GOOGLE_API_KEY - do NOT hardcode it here for deployment.

api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    api_key = st.text_input("Enter your Gemini API Key:", type="YOUR API ")
    if not api_key:
        st.warning("Please enter your Gemini API key to continue.")
        st.stop()

os.environ["GOOGLE_API_KEY"] = api_key


# -------------------------------
# FILE UPLOAD
# -------------------------------
uploaded_file = st.file_uploader("Upload a PDF or TXT document", type=["pdf", "txt"])

if uploaded_file is None:
    st.info("Upload a document to start chatting with it.")
    st.stop()


# -------------------------------
# BUILD PIPELINE (cached so it only runs once per uploaded file)
# -------------------------------
@st.cache_resource(show_spinner="Processing document...")
def build_pipeline(file_bytes, file_name):
    # Save uploaded file temporarily
    temp_path = f"temp_{file_name}"
    with open(temp_path, "wb") as f:
        f.write(file_bytes)

    # Load document
    if temp_path.endswith(".pdf"):
        loader = PyPDFLoader(temp_path)
    else:
        loader = TextLoader(temp_path)

    docs = loader.load()

    # Chunk
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs)

    # Embeddings + Chroma
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(chunks, embeddings)
    chroma_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # BM25
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 3

    # Clean up temp file
    os.remove(temp_path)

    return chroma_retriever, bm25_retriever, len(chunks)


chroma_retriever, bm25_retriever, num_chunks = build_pipeline(
    uploaded_file.getvalue(), uploaded_file.name
)
st.success(f"Document processed into {num_chunks} chunks. Ready to chat!")


# -------------------------------
# HYBRID RETRIEVAL FUNCTION
# -------------------------------
def hybrid_retrieve(query, k=4):
    semantic_results = chroma_retriever.invoke(query)
    keyword_results = bm25_retriever.invoke(query)
    combined = semantic_results + keyword_results

    seen = set()
    unique_chunks = []
    for doc in combined:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_chunks.append(doc)

    return unique_chunks[:k]


# -------------------------------
# LLM
# -------------------------------
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


# -------------------------------
# MEMORY (Streamlit session state)
# -------------------------------
if "history" not in st.session_state:
    st.session_state.history = []  # list of HumanMessage / AIMessage


def format_history(history, max_turns=5):
    recent = history[-(max_turns * 2):]
    formatted = []
    for msg in recent:
        if isinstance(msg, HumanMessage):
            formatted.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            formatted.append(f"Assistant: {msg.content}")
    return "\n".join(formatted)


def rewrite_query_with_context(query, history):
    if not history:
        return query

    history_text = format_history(history)

    rewrite_prompt = f"""Given the conversation history and a follow-up question,
rewrite the follow-up question to be a standalone question that includes
all necessary context. If the follow-up question is already standalone,
return it unchanged. Return ONLY the rewritten question, nothing else.

Conversation History:
{history_text}

Follow-up Question: {query}

Standalone Question:"""

    response = llm.invoke(rewrite_prompt)
    return response.content.strip()


def generate_answer(query, retrieved_chunks, history):
    context = "\n\n".join([chunk.page_content for chunk in retrieved_chunks])
    history_text = format_history(history)

    prompt_text = f"""You are an expert data assistant. Answer the question based ONLY on the provided context.
If the answer is not in the context, strictly say "I don't have enough information in the document."
Do not make up any facts.

Previous Conversation:
{history_text}

Context from Documents:
{context}

Current Question: {query}

Answer:"""

    response = llm.invoke(prompt_text)
    return response.content


# -------------------------------
# CHAT UI
# -------------------------------

# Display past messages
for msg in st.session_state.history:
    role = "user" if isinstance(msg, HumanMessage) else "assistant"
    with st.chat_message(role):
        st.write(msg.content)

# Chat input
user_query = st.chat_input("Ask a question about your document...")

if user_query:
    # Show user message
    with st.chat_message("user"):
        st.write(user_query)

    # Rewrite query using history
    standalone_query = rewrite_query_with_context(user_query, st.session_state.history)

    # Retrieve
    retrieved_chunks = hybrid_retrieve(standalone_query)

    # Generate answer
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer = generate_answer(user_query, retrieved_chunks, st.session_state.history)
        st.write(answer)

        if standalone_query != user_query:
            with st.expander("🔍 Query rewritten for context"):
                st.caption(f"Original: {user_query}")
                st.caption(f"Rewritten: {standalone_query}")

    # Update history
    st.session_state.history.append(HumanMessage(content=user_query))
    st.session_state.history.append(AIMessage(content=answer))


# -------------------------------
# SIDEBAR INFO
# -------------------------------
with st.sidebar:
    st.header("ℹ️ About")
    st.write("""
    **RAG Bot V3.0** combines:
    - 🔍 **Hybrid Retrieval**: BM25 (keyword) + ChromaDB (semantic)
    - 🧠 **Conversational Memory**: Remembers context across turns
    - ⚡ **Gemini 2.5 Flash**: Fast, accurate responses

    Built as part of a 14-day GenAI project series.
    """)

    if st.button("Clear Conversation"):
        st.session_state.history = []
        st.rerun()

ModuleNotFoundError: No module named 'streamlit'